In [ ]:
## googlecolab なら実行
from google.colab import drive
drive.mount("/content/drive/")
%cd "/content/drive/MyDrive/Colab Notebooks/app_labGuide/backend"

In [ ]:
## googlecolab なら実行（研究室サーバの場合は初回のみでOK）
# 1. システムのセットアップ 
!bash ../setup.sh

# 2. Pythonライブラリのインストール
!pip install -r ../requirements.txt

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ==========================================
# 1. データの読み込みと前処理（アンダーサンプリング）
# ==========================================
print("Loading and preprocessing data...")

# wrimeデータセット読み込み
df = pd.read_csv("data/raw/wrime/wrime-ver2.tsv", delimiter="\t")

# 対象とするプルチックの8感情カラム（書き手の感情強度 0〜3）
# emotion_cols = ["Writer_Joy", "Writer_Trust", "Writer_Fear", "Writer_Surprise", "Writer_Sadness", "Writer_Disgust", "Writer_Anger", "Writer_Anticipation"]
emotion_cols = ['Avg. Readers_Joy', 'Avg. Readers_Sadness', 'Avg. Readers_Anticipation', 'Avg. Readers_Surprise', 'Avg. Readers_Anger', 'Avg. Readers_Fear', 'Avg. Readers_Disgust', 'Avg. Readers_Trust']

# 全感情の強度の合計を計算
df['total_emotion'] = df[emotion_cols].sum(axis=1)

# 【アンダーサンプリング】合計が0（完全な中立）のサンプルを間引く
print(f"Original data size: {len(df)}")
df_filtered = df[df['total_emotion'] > 0].copy()
print(f"Filtered data size (dropped all-zeros): {len(df_filtered)}")

# 入力テキストと、8次元のラベルリストを作成
df_filtered['labels'] = df_filtered[emotion_cols].astype(float).values.tolist()
dataset = Dataset.from_pandas(df_filtered[['Sentence', 'labels']]) # Sentenceは入力テキストのカラム名

# 訓練・検証データの分割
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# ==========================================
# 2. トークナイズ処理
# ==========================================
model_name = "ku-nlp/deberta-v2-base-japanese"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# ==========================================
# 3. モデルとカスタムTrainerの定義（重み付き損失関数）
# ==========================================
# problem_type="regression" と num_labels=8 を指定して回帰モデルとしてロード
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=8,
    problem_type="regression"
)

class WeightedMSETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits # 予測された感情強度 (batch_size, 8)

        # 損失関数 (MSE) を reduction='none' で計算し、要素ごとの誤差を取得
        loss_fct = nn.MSELoss(reduction='none')
        loss = loss_fct(logits, labels)

        # 【重み付け】正解ラベルが0より大きい場合は重み2.0、0の場合は1.0
        weight = torch.where(labels > 0, torch.tensor(2.0).to(labels.device), torch.tensor(1.0).to(labels.device))

        # 重みを掛けて平均をとる
        weighted_loss = (loss * weight).mean()

        return (weighted_loss, outputs) if return_outputs else weighted_loss

# ==========================================
# 4. 学習の実行
# ==========================================
training_args = TrainingArguments(
    output_dir="models/deberta-emotion-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True, # GPUでの高速化
    report_to="none"
)

trainer = WeightedMSETrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

print("Starting training...")
trainer.train()

# モデルの保存
trainer.save_model("models/deberta-emotion-classifier-best")
tokenizer.save_pretrained("models/deberta-emotion-classifier-best")
print("Training complete and model saved.")